# 04 · Superposition as a phase change

**Paper:** "Superposition as a Phase Change", pp. 14–16; Comments & Replications (McGrath),
pp. 51–53.
**Experiment:** E02 `phase_change`.

**Learning goals**

1. Label what a trained model did with an "extra" feature: dropped it, stored it in
   superposition, gave it a dedicated dimension, or confused it with another feature.
2. Read the empirical and theoretical phase diagrams side by side, and know how each is made:
   a vote of nine trained models per cell, and the cheapest of a few exactly solved weight
   families (notebook 03).
3. Find where the two disagree, and tell a real disagreement from a cell on a boundary or a
   model stuck in a worse minimum (McGrath, p. 52).
4. See why McGrath's "confused feature" solution appears only when biases are ignored.

The E02 quick profile takes about 8 minutes on a laptop CPU.

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

In [ ]:
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Image, display

from superposition import lab
from superposition.experiments.phase_change import extra_feature_regime
from superposition.experiments.runner import run_experiment
from superposition.runs import list_runs

pd.set_option("display.max_colwidth", 120)

REUSE = True  # reuse your latest quick-profile run if there is one; False always re-runs


def quick_run(name: str):
    """Your latest complete quick-profile run of ``name``, or a fresh quick run."""
    if REUSE:
        for run in reversed(list_runs(name)):
            if yaml.safe_load(run.config_path.read_text())["profile"] == "quick":
                print(f"reusing run {run.run_id}")
                return run
    return run_experiment(name, "quick", device="auto")


def criteria(run) -> pd.DataFrame:
    doc = run.read_acceptance()
    print("overall:", doc["overall"], f"({doc['n_passed']}/{doc['n_graded']} graded criteria)")
    return pd.DataFrame(doc["criteria"])[["group", "id", "status", "value"]]

## 1. What happened to the extra feature?

Two features in one dimension (p. 14). Feature 0 has importance 1; feature 1 is the extra one.
A trained W is a 1 × 2 row, so each feature is just a signed number. **Predict** the label of
each model below before running the cell: W = [1, 0], [0, 1], [1, −1] and [0.9, 0.9].

In [ ]:
W = torch.tensor([[[1.0, 0.0]], [[0.0, 1.0]], [[1.0, -1.0]], [[0.9, 0.9]]])
labels, norm, score = extra_feature_regime(W, feature=1)
pd.DataFrame(
    {
        "W": [w[0].tolist() for w in W],
        "‖W_extra‖": norm.numpy().round(3),
        "Σⱼ (Ŵ_extra·Wⱼ)², represented j": score.numpy().round(3),
        "label": labels,
    }
)

The rule (docs/DECISIONS.md D-304): the extra feature is *not learned* if ‖W_extra‖ ≤ 0.5;
*dedicated* if it is represented and orthogonal to every other represented feature; otherwise
it shares a direction with another feature, *superposition* if they point opposite ways (an
antipodal pair) and *confused* if they point the same way. The last one is McGrath's solution
W₁ ≈ W₂ (p. 52): both outputs light up whenever either feature is on.

## 2. The empirical and theoretical diagrams (pp. 15–16)

E02 trains ten models per cell of a 12 × 12 grid (relative importance of the extra feature ×
feature density), for n = 2, m = 1 and for n = 3, m = 2. Per cell it drops the highest-loss
model and takes the majority label of the other nine (p. 14). Beside it, `superposition.theory`
solves each weight family exactly and keeps the cheapest (notebook 03, section 4).

**Predict:** where on the grid do you expect each regime? Where would you expect the empirical
diagram to disagree with the theory?

In [ ]:
run = quick_run("phase_change")
criteria(run)

In [ ]:
for label in ("n2_m1", "n3_m2"):
    display(Image(filename=str(run.figures_dir / f"phase_{label}.png")))

Both diagrams show the three regions where pp. 15–16 draw them: an unimportant extra feature is
dropped when features are dense (top left), an important one gets its own dimension (top right),
and once features are sparse it is stored in superposition almost everywhere. The criteria table
gives the agreement. The disagreeing cells follow, with each cell's vote (`counts`): look at how
close the votes are.

In [ ]:
cells = run.read_table("cells")
columns = [
    "setting",
    "density",
    "importance",
    "empirical",
    "theory",
    "best_model",
    "counts",
    "theory_relative_gap",
    "near_theory_boundary",
]
disagree = cells[~cells["agree"]][columns]
print(disagree.groupby("setting").size().rename("disagreeing cells").to_string())
disagree.round(4)

## 3. Stuck models, or solutions the theory misses?

A disagreeing cell can mean two different things. Most models may be stuck in a worse minimum,
which is no evidence against the theory. Or the trained models may have found something better
than every family the theory considers, which *is* evidence that the families are incomplete.
The run's final losses cannot tell these apart: each is an independent 20,000-sample estimate,
and the lowest of nine noisy estimates is biased low.

`lab.phase_cell_shared_losses` does better. It gives the exact loss (notebook 03) of every
model whose outputs each read at most one other feature: all the theory's families, and every
trained n = 2, m = 1 model. For the rest it scores all models on one common sample of inputs.
That sample makes *differences* precise when the models pay on the same inputs. It does not
when they pay on different rare events: at density 0.01 the loss comes from a few hundred
double activations, and pairing the extra feature with feature 0 or with feature 1 is paid for
on different ones. Treat sampled comparisons at the lowest densities as indicative only.

First, one cell. (The lab's **Phase diagrams** page shows the models and families of any cell
you click.)

In [ ]:
data = lab.load_phase_change(run)


def cell_index(problem, density, importance):
    return (
        int(np.argmin(np.abs(problem.density - density))),
        int(np.argmin(np.abs(problem.importance - importance))),
    )


problem = data.problems["n2_m1"]
rows = disagree[disagree["setting"] == "n2_m1"]
first = rows.iloc[0] if len(rows) else cells[cells["setting"] == "n2_m1"].iloc[78]
d, i = cell_index(problem, first["density"], first["importance"])
print(f"cell: density {problem.density[d]:.3g}, relative importance {problem.importance[i]:.3g}")
lab.phase_cell_shared_losses(problem, d, i, data.discard_worst)

Now every disagreeing cell: how does the best voting model compare with the theory's winning
family? For n = 2 the comparison is exact; for n = 3 it uses 2,000,000 shared samples. A ratio
near 1 means the best model *is* the theory's solution and the vote was outnumbered by worse
models; a ratio clearly below 1 means the models found something the families do not contain.
This takes about a minute.

In [ ]:
rows = []
for label, problem in data.problems.items():
    for r in disagree[disagree["setting"] == label].to_dict("records"):
        d, i = cell_index(problem, r["density"], r["importance"])
        table = lab.phase_cell_shared_losses(problem, d, i, data.discard_worst, n_samples=2_000_000)
        families = table[table["source"].str.startswith("theory")]
        winner = families.loc[families["exact loss"].idxmin()]
        models = table[table["source"].str.startswith("trained")]
        key = "exact loss" if bool(models["exact loss"].notna().all()) else "shared-sample loss"
        best = models.loc[models[key].idxmin()]
        rows.append(
            {
                "setting": label,
                "density": r["density"],
                "importance": r["importance"],
                "vote": r["counts"],
                "theory": r["theory"],
                "best model": best["regime"],
                "compared on": key,
                "best model / theory's optimum": best[key] / winner[key],
                "best model W": best["W"],
            }
        )
verdict = pd.DataFrame(rows)
verdict["same label"] = verdict["best model"] == verdict["theory"]
verdict["below the theory"] = verdict["best model / theory's optimum"] < 0.99
print(verdict.groupby("setting")[["same label", "below the theory"]].sum())
verdict.round(4)

In the cited quick run (docs/REPRODUCTION.md, E02) the lowest-loss voting model has the
theory's label in every disagreeing cell, and matches its optimum to within about half a
percent in most of them: the disagreements are votes lost to models stuck in worse minima.
Four n = 2 cells are different. There the best model is 5–7% *below* the theory, and its W is
an asymmetric pair, the more important feature longer (about 1.02 against 0.86). The family
W ⊥ [1, 1] cannot express it, because W₁ + W₂ = 0 forces equal lengths. It is still labeled
superposition, so the diagram does not change, but the paper's families are not the whole
solution space. (The lab's **Phase diagrams** page shows these cells' models when you click
them.)

## 4. McGrath's confused feature (pp. 51–53)

McGrath solved the n = 2, m = 1 problem exactly while "ignoring bias terms" and found a regime
where both features share one direction with the same sign. The theory module can do both:
optimize every bias, or fix every bias at 0.

In [ ]:
display(Image(filename=str(run.figures_dir / "theory_variants_n2_m1.png")))
confused = [c for c in run.read_acceptance()["criteria"] if c["id"] == "confused_regime"]
pd.DataFrame([{"setting": c["group"], **c["value"]} for c in confused])

With biases fixed at 0 the confused family wins a band of cells; once biases are free it never
does. A bias can predict a feature's mean on its own, which removes most of what the shared
direction was buying. Whether trained models (which always have biases) ever end up confused is
the "cells_with_a_confused_model" column.

Finally, each family's optimal loss along density at three importances: the curves cross, and
the kink in their minimum is the first-order phase change of p. 15.

In [ ]:
display(Image(filename=str(run.figures_dir / "family_losses_n2_m1.png")))

## Exercises

**1.** Which cell of the n = 2 diagram has the smallest relative gap between the two best
theoretical families? How did its nine models vote?

<details><summary>Solution</summary>

```python
n2 = cells[cells["setting"] == "n2_m1"]
row = n2.loc[n2["theory_relative_gap"].idxmin()]
print(row[["density", "importance", "theory", "empirical", "counts", "theory_relative_gap"]])
```

Cells with a small gap sit on a theoretical boundary, where two solutions are nearly as good and
training can land in either.
</details>

**2.** Recompute the agreement if each cell is labeled by its single lowest-loss model instead of
the vote. Is it higher or lower, and why might that be?

<details><summary>Solution</summary>

```python
for setting, group in cells.groupby("setting"):
    vote = (group["empirical"] == group["theory"]).mean()
    best = (group["best_model"] == group["theory"]).mean()
    print(f"{setting}: vote {vote:.3f}, lowest-loss model {best:.3f}")
```

The lowest-loss model is the one most likely to have found the global optimum, so where the
vote is dragged by stuck models the best model agrees more often. The run also records this as
the INFO criterion `best_model_agreement`.
</details>

**3.** Train the ten models of the first cell of section 3 yourself, for three times as many
steps.
Do the stuck models escape?

<details><summary>Solution</summary>

```python
from superposition.config import TrainConfig
from superposition.data import IndependentSparse
from superposition.models import TiedModel
from superposition.train import train_from_config

problem = data.problems["n2_m1"]
d, i = cell_index(problem, first["density"], first["importance"])
p, extra = float(problem.density[d]), float(problem.importance[i])
model = TiedModel.from_kind("relu_output", 2, 1, 10, seed=0)
same_sign = (model.W[:, 0, 0] * model.W[:, 0, 1] > 0).detach().numpy()  # before training
config = TrainConfig(steps=30_000, batch_size=256, eval_samples=100_000)
result = train_from_config(
    model, IndependentSparse(p, 10, 2), torch.tensor([1.0, extra]), config, seed=0
)
labels, _, _ = extra_feature_regime(model.W.detach(), feature=1)
pd.DataFrame(
    {"loss": result.final_loss.numpy(), "label": labels, "started with the same sign": same_sign}
).sort_values("loss")
```

In the cited run's first disagreeing cell, every model whose two weights start with the same
sign still has the extra feature dropped after 30,000 steps, and every model that starts with
opposite signs finds the antipodal pair. The same-sign start is a basin that longer training
does not leave: stuck models, not a wrong theory.
</details>

**4.** At a dense cell with the extra feature about as important as the other, compare each
family's loss with and without biases. What does a bias buy the dropped feature?

<details><summary>Solution</summary>

```python
problem = data.problems["n2_m1"]
d_dense = len(problem.density) - 1
i_mid = int(np.argmin(np.abs(np.log(problem.importance))))
lab.phase_cell_theory(problem, d_dense, i_mid)
```

A dropped feature's output is ReLU(b): with b = p/2 it predicts the feature's mean and loses only
the variance p/3 − p²/4, instead of its whole second moment p/3. Without biases that saving is
gone, and sharing a direction (the confused family) becomes the cheapest way to predict
something for both features.
</details>